# 1. Подготовьте текстовые и категориальные признаки

Текст очистите и преобразуйте с помощью `TfidfVectorizer`, а категориальные признаки закодируйте через `DictVectorizer`.


In [1]:
import re
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge

train = pd.read_csv('salary-train.csv')
test  = pd.read_csv('salary-test-mini.csv')
y_train = train['SalaryNormalized']
train['FullDescription'] = train['FullDescription'].apply(lambda t: re.sub('[^a-zA-Z0-9]', ' ', t.lower()))
test['FullDescription'] = test['FullDescription'].apply(lambda t: re.sub('[^a-zA-Z0-9]', ' ', t.lower()))

tfidf = TfidfVectorizer(min_df=5)
X_train_text = tfidf.fit_transform(train['FullDescription'])
X_test_text  = tfidf.transform(test['FullDescription'])

train['LocationNormalized'] = train['LocationNormalized'].fillna('nan')
train['ContractTime'] = train['ContractTime'].fillna('nan')
test['LocationNormalized'] = test['LocationNormalized'].fillna('nan')
test['ContractTime'] = test['ContractTime'].fillna('nan')

enc = DictVectorizer()
X_train_categ = enc.fit_transform(train[['LocationNormalized', 'ContractTime']].to_dict('records'))
X_test_categ = enc.transform(test[['LocationNormalized', 'ContractTime']].to_dict('records'))

X_train = hstack([X_train_text, X_train_categ])
X_test = hstack([X_test_text, X_test_categ])


# 2. Обучите Ridge Regression и получите прогнозы

Используйте `Ridge(alpha=1)` и выведите предсказания для тестовых объектов.


In [2]:
ridge = Ridge(alpha=1)
ridge.fit(X_train, y_train)
pred=ridge.predict(X_test)
print(round(pred[0], 2), round(pred[1], 2))


56572.72 37196.93
